Configurando ambiente de trabalho

In [0]:
%sql
use catalog stack_overgol;

In [0]:
from pyspark.sql.functions import (
    col, lower, trim, when, regexp_extract, regexp_replace, length, concat, lit, split, explode, substring, coalesce, try_to_date, expr, initcap, ceil, unix_timestamp
)

1 - Clientes

In [0]:
df = spark.table("bronze.clientes")

df = df.withColumn(
    "origem",
    when(lower(trim(col("origem"))).isin("web"), "Web")
    .when(lower(trim(col("origem"))).isin("app"), "App")
    .when(lower(trim(col("origem"))).isin("indicação", "indicacao"), "Indicação")
    .otherwise("Outro")
)

df = df.withColumn(
    "ramal",
    regexp_extract(col("telefone"), r"(?i)r\.?\s*(\d+)", 1)
)

df = df.withColumn(
    "ramal",
    when(col("ramal") == "", None).otherwise(col("ramal"))
)

df = df.withColumn(
    "telefone_limpo",
    trim(
        regexp_replace(
            regexp_replace(
                regexp_replace(col("telefone"), r"(?i)r\.?\s*\d+", ""),
                r"\D", ""
            ),
            r"^55(?=\d{10,11}$)", ""
        )
    )
)

df = df.withColumn(
    "telefone_formatado",
    when(length(col("telefone_limpo")) == 10,
        concat(
            lit("("), substring("telefone_limpo", 1, 2), lit(") "),
            substring("telefone_limpo", 3, 4), lit("-"),
            substring("telefone_limpo", 7, 4)
        )
    ).when(length(col("telefone_limpo")) == 11,
        concat(
            lit("("), substring("telefone_limpo", 1, 2), lit(") "),
            substring("telefone_limpo", 3, 5), lit("-"),
            substring("telefone_limpo", 8, 4)
        )
    ).otherwise(None) 
)

df = df.withColumn(
    "email_tratado",
    lower(trim(col("email")))
)

df = df.withColumn(
    "email_tratado",
    when(
        col("email_tratado").isNotNull() & (~col("email_tratado").contains("@")),
        regexp_replace(
            col("email_tratado"),
            r"(gmail|yahoo|hotmail|outlook|uol)",
            r"@\1"
        )
    ).otherwise(col("email_tratado"))
)

df = df.withColumn(
    "email_tratado",
    regexp_replace(col("email_tratado"), r"@{2,}", "@")
)

df = df.withColumn("nome", initcap(col("nome")))
df = df.withColumn("sobrenome", initcap(col("sobrenome")))

df = df.dropDuplicates(["id_cliente"])

df = df.select( 
    col("id_cliente").cast("string"), 
    col("nome").cast("string").alias("nome_cliente"),
    col("sobrenome").cast("string").alias("sobrenome_cliente"), 
    col("email_tratado").alias("email_cliente"),
    col("telefone_formatado").alias("telefone_cliente"),
    col("ramal").alias("ramal_cliente"),
    col("genero").cast("string").alias("genero_cliente"), 
    col("data_nascimento").cast("date").alias("data_nascimento_cliente"),
    col("data_cadastro").cast("date").alias("data_cadastro_cliente"),
    col("endereco").cast("string").alias("endereco_cliente"),
    col("cidade").cast("string").alias("cidade_cliente"), 
    col("estado").cast("string").alias("estado_cliente"), 
    col("pais").cast("string").alias("pais_cliente"),
    col("origem").alias("origem_cliente")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes")

2 - Dispositivos por cliente

In [0]:
df = spark.table("bronze.clientes")

df_dispositivo = df.select(
    col("id_cliente").cast("string"),
    explode(
        split(col("device_ids"), ";")
    ).alias("id_dispositivo")
)

df_dispositivo = df_dispositivo.withColumn(
    "id_dispositivo",
    trim(col("id_dispositivo"))
)

df_dispositivo = df_dispositivo.filter(col("id_dispositivo") != "")

df_dispositivo = df_dispositivo.dropDuplicates(["id_cliente", "id_dispositivo"])

df_dispositivo.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clientes_dispositivo")

3 - Catálogos Produto

In [0]:
df = spark.table("bronze.catalogo_produtos")

df = df.withColumn(
    "ativo",
    when(
        lower(trim(col("ativo"))).isin("s", "sim", "yes", "1"),
        True
    ).when(
        lower(trim(col("ativo"))).isin("n", "nao", "não", "no", "0"),
        False
    ).otherwise(None)
)

df = df.withColumn(
    "categoria_tratada",
    lower(trim(col("categoria")))
)

df = df.withColumn(
    "categoria_tratada",
    regexp_replace(col("categoria_tratada"), r"[0-9@]", "")
)

df = df.withColumn(
    "categoria_tratada",
    when(col("categoria_tratada").isin(
        "eletronicos","eletronico","elet","eletronico"
    ), "Eletrônicos")

    .when(col("categoria_tratada").isin(
        "vestuario","vestu","vest","vestuarios"
    ), "Vestuário")

    .when(col("categoria_tratada").isin(
        "casa","cas"
    ), "Casa")

    .when(col("categoria_tratada").isin(
        "esportes","esporte","esport","esp"
    ), "Esportes")

    .when(col("categoria_tratada").isin(
        "beleza","bel","belz","beleza"
    ), "Beleza")

    .when(col("categoria_tratada").isin(
        "automotivo","autom","aut","automotiv"
    ), "Automotivo")

    .when(col("categoria_tratada").isin(
        "brinquedo","brinquedos","brin","brinq"
    ), "Brinquedos")

    .when(col("categoria_tratada").isin(
        "moveis","mov","moveis","mveis"
    ), "Móveis")

    .otherwise("Outros")
)

df = df.withColumn(
    "peso_kg",
    when(lower(trim(col("peso_kg"))) == "null", None)
    .otherwise(col("peso_kg").cast("double"))
)

df = df.withColumn(
    "estoque_disponivel",
    when(
        lower(trim(col("estoque_disponivel"))) == "null",
        0
    ).otherwise(col("estoque_disponivel").cast("int"))
)

df = df.withColumn(
    "preco_tratado",
    when(
        lower(trim(col("preco"))) == "null",
        None
    ).otherwise(col("preco"))
)

df = df.withColumn(
    "preco_tratado",
    regexp_replace(col("preco_tratado"), r"[R$\s]", "")
)

df = df.withColumn(
    "preco_tratado",
    regexp_replace(col("preco_tratado"), ",", ".")
)

df = df.withColumn(
    "preco_tratado",
    col("preco_tratado").cast("double")
)

df = df.withColumn(
    "preco_tratado",
    when(col("preco_tratado") <= 0, None)
    .otherwise(col("preco_tratado"))
)

df = df.dropDuplicates(["id_produto"])

df = df.select( 
    col("id_produto").cast("string"), 
    col("nome_produto").cast("string"),
    col("categoria_tratada").cast("string").alias("categoria_produto"),
    col("preco_tratado").alias("preco_produto"),
    col("fornecedor").cast("string").alias("fornecedor_produto"),
    col("peso_kg").alias("peso_kg_produto"),
    col("estoque_disponivel").alias("estoque_produto"),
    col("ativo").cast("boolean").alias("produto_ativo"),
    col("data_cadastro_produto").cast("date")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.catalogo_produtos")

4 - Trilha de ações do usuário

In [0]:
df = spark.table("bronze.clickstream")

df = df.withColumn(
    "tipo_normalizado",
    lower(trim(col("tipo_evento")))
)

df = df.withColumn(
    "tipo_normalizado",
    regexp_replace(col("tipo_normalizado"), r"[-\s]", "_")
)

df = df.withColumn(
    "tipo_padronizado",
    
    when(col("tipo_normalizado").isin(
        "pageview","page_view","page_view","pv"
    ), "Vizualização de Página")

    .when(col("tipo_normalizado").isin(
        "add_to_cart","addtocart","adicionar"
    ), "Adicionar no Carrinho")

    .when(col("tipo_normalizado").isin(
        "login","log_in","signin"
    ), "Login")

    .when(col("tipo_normalizado").isin(
        "purchase","compra","buy"
    ), "Compra")

    .when(col("tipo_normalizado").isin(
        "checkout","check_out","pagamento"
    ), "Pagamento")

    .when(col("tipo_normalizado").isin(
        "search","srch","busca"
    ), "Busca")

    .when(col("tipo_normalizado").isin(
        "abandon_cart","abandono","abandon"
    ), "Abandono de Carrinho")

    .otherwise("Outros")
)

df = df.withColumn(
    "canal_normalizado",
    lower(trim(col("canal")))
)

df = df.withColumn(
    "canal_normalizado",
    regexp_replace(col("canal_normalizado"), r"[\s\-]", "_")
)

df = df.withColumn(
    "canal_padronizado",
    when(col("canal_normalizado").isin(
        "web", "mobile_web"
    ), "Web")
    
    .when(col("canal_normalizado").isin(
        "app", "aplicativo"
    ), "App")
    
    .otherwise("Outros")
)

df = df.withColumn(
    "dispositivo_normalizado",
    lower(trim(col("dispositivo")))
)

df = df.withColumn(
    "dispositivo_padronizado",
    
    when(col("dispositivo_normalizado").isin(
        "desktop", "computador"
    ), "Desktop")
    
    .when(col("dispositivo_normalizado").isin(
        "mobile", "celular", "mob"
    ), "Mobile")
    
    .when(col("dispositivo_normalizado").isin(
        "tablet", "tab"
    ), "Tablet")
    
    .otherwise("Outros")
)

df = df.withColumn(
    "origem_sessao_tratada",
    lower(trim(col("origem_sessao")))
)

df = df.withColumn(
    "origem_sessao_tratada",
    when(col("origem_sessao_tratada") == "social", "Social")
    .when(col("origem_sessao_tratada") == "organic", "Orgânico")
    .when(col("origem_sessao_tratada") == "paid_search", "Busca Paga")
    .when(col("origem_sessao_tratada") == "direct", "Direto")
    .when(col("origem_sessao_tratada") == "email", "Email")
    .otherwise("Outros")
)

df = df.withColumn(
    "tempo_pagina_seg",
    when(col("tempo_pagina_seg") < 0, None)
    .otherwise(col("tempo_pagina_seg"))
)

df = df.dropDuplicates(["id_evento"])

df = df.select( 
    col("id_evento").cast("string"), 
    col("id_sessao").cast("string"),
    col("id_cliente").cast("string"), 
    col("id_dispositivo").cast("string"), 
    col("id_produto").cast("string"),
    col("tipo_padronizado").cast("string").alias("tipo_evento"), 
    col("canal_padronizado").cast("string").alias("canal_evento"), 
    col("dispositivo_padronizado").cast("string").alias("dispositivo_evento"),
    col("origem_sessao_tratada").cast("string").alias("origem_sessao"),
    col("data_evento").cast("timestamp"),
    col("tempo_pagina_seg").cast("int") 
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.clickstream")

5 - Pedidos

In [0]:
df = spark.table("bronze.pedidos")

df = df.withColumn(
    "data_pedido",
    coalesce(
        try_to_date(col("data_pedido"), "yyyy-MM-dd"),
        try_to_date(col("data_pedido"), "yyyy/MM/dd"),
        try_to_date(col("data_pedido"), "yyyy/dd/MM"),
        try_to_date(col("data_pedido"), "dd/MM/yyyy"),
        try_to_date(col("data_pedido"), "MM-dd-yyyy")
    )
)

df = df.withColumn(
    "valor_pedido",
    expr("""
        try_cast(
            regexp_replace(
                regexp_replace(trim(valor_pedido), 'R\\$\\s*', ''),
                ',', '.'
            ) as double
        )
    """)
)

df = df.withColumn(
    "valor_pedido",
    when(col("valor_pedido") <= 0, None)
    .otherwise(col("valor_pedido"))
)

df = df.withColumn(
    "metodo_pagamento",
    lower(regexp_replace(col("metodo_pagamento"), "[^a-zA-Z0-9]", ""))
)

df = df.withColumn(
    "metodo_pagamento",
    when(col("metodo_pagamento").isin("pix", "p1x", "plx"), "Pix")
    .when(col("metodo_pagamento").isin("boleto", "b0leto", "bol", "crt"), "Boleto")
    .when(col("metodo_pagamento").isin("cartao", "crtao"), "Cartão")
    .otherwise("Outros")
)

df = df.withColumn(
    "quantidade",
    when(lower(trim(col("quantidade"))) == "um", "1")
    .when(lower(trim(col("quantidade"))) == "quatro", "4")
    .when(lower(trim(col("quantidade"))) == "cinco", "5")
    .otherwise(col("quantidade"))
)

df = df.withColumn(
    "quantidade",
    expr("try_cast(quantidade as double)")
)

df = df.withColumn(
    "quantidade",
    when(col("quantidade") < 0, None)
    .otherwise(col("quantidade").cast("int"))
)

df = df.withColumn(
    "status_tratado",
    lower(trim(col("status")))
)

df = df.withColumn(
    "status_tratado",
    when(col("status_tratado").isin("aprovado", "aprovadoo", "aprov", "apr"), "Aprovado")
    .when(col("status_tratado").isin("reembolsado", "reembolso", "reemb", "reembolsad", "reembolsadoo"), "Reembolsado")
    .when(col("status_tratado").isin("recusado", "recus", "recusadoo", "rec"), "Recusado")
    .when(col("status_tratado").isin("process", "processando", "proc"), "Processando")
    .otherwise("Outros")
)

df = df.dropDuplicates(["id_pedido"])

df = df.select( 
    col("id_pedido").cast("string"), 
    col("id_cliente").cast("string"),
    col("id_produto").cast("string"),
    col("valor_pedido"),
    col("data_pedido"),
    col("metodo_pagamento"),
    col("status_tratado").cast("string").alias("status_pedido"),
    col("quantidade").alias("quantidade_produto")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.pedidos")

6 - Avaliações

In [0]:
df = spark.table("bronze.avaliacoes")

df = df.withColumn(
    "recomenda",
    when(
        lower(trim(col("recomenda"))).isin("s", "sim", "yes", "1"),
        True
    ).when(
        lower(trim(col("recomenda"))).isin("n", "nao", "não", "no", "0"),
        False
    ).otherwise(None)
)

df = df.withColumn(
    "nota_produto",
    when(lower(trim(col("nota_produto"))) == "ótimo", 5)
    .when(lower(trim(col("nota_produto"))) == "bom", 4)
    .when(lower(trim(col("nota_produto"))) == "ruim", 2)
    .when(lower(trim(col("nota_produto"))) == "péssimo", 1)
    .otherwise(col("nota_produto").cast("int"))
)

df = df.withColumn(
    "nota_produto",
    when(col("nota_produto") < 1, 1)
    .when(col("nota_produto") > 5, 5)
    .otherwise(col("nota_produto"))
)

df = df.withColumn(
    "nota_nps",
    when(lower(trim(col("nota_nps"))) == "ótimo", 10)
    .when(lower(trim(col("nota_nps"))) == "bom", 8)
    .when(lower(trim(col("nota_nps"))) == "ruim", 2)
    .when(lower(trim(col("nota_nps"))) == "péssimo", 0)
    .otherwise(col("nota_nps").cast("int"))
)

df = df.withColumn(
    "nota_nps",
    when(col("nota_nps") < 0, 0)
    .when(col("nota_nps") > 10, 10)
    .otherwise(col("nota_nps"))
)

df = df.withColumn(
    "data_avaliacao",
    coalesce(
        try_to_date(col("data_avaliacao"), "yyyy-MM-dd HH:mm:ss"),
        try_to_date(col("data_avaliacao"), "yyyy/dd/MM HH:mm:ss")
    )
)

pedidos = spark.table("silver.pedidos").select(
    col("id_pedido"),
    col("data_pedido")
)

df = df.join(pedidos, on="id_pedido", how="left")

df = df.withColumn(
    "data_avaliacao",
    coalesce(col("data_avaliacao"), col("data_pedido"))
)

df = df.drop("data_pedido")

df = df.dropDuplicates(["id_avaliacao"])

df = df.select( 
    col("id_avaliacao").cast("string"), 
    col("id_pedido").cast("string"),
    col("id_cliente").cast("string"),
    col("id_produto").cast("string"),
    col("nota_produto"),
    col("comentario").cast("string").alias("comentario_avaliacao"),
    col("nota_nps"),
    col("recomenda").alias("recomenda_produto"),
    col("data_avaliacao")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.avaliacoes")

7 - Tickets Suporte

In [0]:
df = spark.table("bronze.suporte_tickets")

df = df.withColumn(
    "tipo_problema",
    when(
        lower(trim(col("tipo_problema"))).isin(
            "pro","produto","p3oduto","produto","prod","product","produto"
        ), "Produto"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "pag","pagamento","p4gamento","pay","payment"
        ), "Pagamento"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "entrega","3ntrega","entr","ent","delay","del"
        ), "Entrega"
    ).when(
        lower(trim(col("tipo_problema"))).isin(
            "reembolso","reemb","r3embolso","refund","ref","reembolso"
        ), "Reembolso"
    ).otherwise("Outro")
)

df = df.withColumn(
    "tempo_resolucao_horas",
    ceil(
        (unix_timestamp(col("data_resolucao")) - unix_timestamp(col("data_abertura"))) / 3600
    ).cast("int")
)

df = df.dropDuplicates(["ticket_id"])

df = df.select( 
    col("ticket_id").cast("string"), 
    col("id_cliente").cast("string"),
    col("id_pedido").cast("string"),
    col("tipo_problema"),
    col("data_abertura").cast("timestamp"),
    col("data_resolucao").cast("timestamp"),
    col("tempo_resolucao_horas"),
    col("agente_suporte").cast("string"),
    col("nota_avaliacao").cast("int").alias("nota_avaliacao_problema")
)

df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.suporte_tickets")